# Run the five-record human-reviewed pilot in Colab

Commit/push the current code before running this notebook. The reviewed gold is
private and excluded from Git: upload `halueval_reviewed_pilot_inputs.zip` separately.
Add your API key in Colab **Secrets** as `GROQ_API_KEY`, granting this notebook access.

This profile uses a streaming **FTS5 lexical index** for pipeline+B0; the existing
BM25 profiles are unchanged. Report this retrieval method with pilot results.
No GPU is needed for prompted verification/repair. Use Python 3.12 for the pinned
annotation environment. Indexing the full FEVER dump needs sufficient local and
Drive storage. No gold evidence is fed to generation.

In [ ]:
from pathlib import Path
import sys, os, subprocess, json, shutil, zipfile
REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
CODE_REF = ""  # Prefer the exact commit you pushed; empty uses the cloned branch.
REPO = Path("/content/hallucination_project")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if CODE_REF:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REF], check=True)
assert (REPO / "requirements-evaluation.txt").is_file(), "Push the current code, then clone that revision."
assert sys.version_info[:2] == (3, 12), "Use a Python 3.12 environment for the pinned spaCy stack."
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
ENV = Path("/content/reviewed_pilot_env")
if not ENV.exists():
    subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO / "requirements-evaluation.txt")], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
os.chdir(REPO)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## Upload reviewed annotations
Upload the generated ZIP. This copies the five reviewed files without displaying
answers, citations, or credentials in the notebook output.

In [ ]:
from google.colab import files, drive, userdata
uploads = files.upload()
archives = [name for name in uploads if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly the reviewed-input ZIP."
import io
required = {"pilot_dev.jsonl", "pilot_dev.sources.json", "pilot_dev.manifest.json", "pilot_splits.json", "pilot_annotation_review.md"}
GOLD = REPO / "data/gold"
GOLD.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploads[archives[0]])) as archive:
    assert set(archive.namelist()) == required, "Unexpected reviewed-input archive contents."
    for name in sorted(required):
        data = archive.read(name)
        target = GOLD / name
        if target.exists():
            assert target.read_bytes() == data, "Existing gold differs; use a new checkout/version."
        else:
            target.write_bytes(data)
import hashlib
review_manifest = json.loads((GOLD / "pilot_dev.manifest.json").read_text())
for name, expected_hash in review_manifest["files_sha256"].items():
    assert hashlib.sha256((GOLD / name).read_bytes()).hexdigest() == expected_hash, "Reviewed packet checksum mismatch"
# Secrets are inherited by child processes but never printed or written to code.
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "Add GROQ_API_KEY to Colab Secrets."
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/hallucination_reviewed_pilot")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## Build or reuse the full FEVER FTS5 index
The index is built by streaming pages into SQLite, without retaining a BM25
corpus in RAM. Keep `MAX_PASSAGES=None` for the full corpus. A passage cap produces
only a diagnostic subset and must be recorded as such. Existing partial builds
are not automatically overwritten; inspect them before rebuilding.

In [ ]:
import urllib.request
INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_INDEX = DRIVE_ROOT / "fever_fts.sqlite"
MAX_PASSAGES = None
INDEX.parent.mkdir(parents=True, exist_ok=True)
if not INDEX.exists() and DRIVE_INDEX.exists():
    shutil.copyfile(DRIVE_INDEX, INDEX)
if not INDEX.exists():
    archive = Path("/content/fever-wiki-pages.zip")
    if not archive.exists():
        part = archive.with_suffix(".zip.part")
        urllib.request.urlretrieve("https://s3-eu-west-1.amazonaws.com/fever.ai/wiki-pages.zip", part)
        part.replace(archive)
    extract_root = Path("/content/fever_wiki")
    wiki = extract_root / "wiki-pages"
    extraction_marker = extract_root / ".extraction_complete"
    if not extraction_marker.exists() or extraction_marker.read_text() != str(archive.stat().st_size):
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                destination = (extract_root / member.filename).resolve()
                assert destination.is_relative_to(extract_root.resolve()), "Unsafe archive path"
            zipped.extractall(extract_root)
        extraction_marker.write_text(str(archive.stat().st_size))
    assert any(wiki.glob("wiki-*.jsonl")), "FEVER wiki-page extraction is incomplete."
    command = [PYTHON, "scripts/build_fever_fts.py", "--wiki-pages-dir", str(wiki), "--output", str(INDEX)]
    if MAX_PASSAGES is not None:
        command += ["--max-passages", str(MAX_PASSAGES)]
    subprocess.run(command, check=True, cwd=REPO)
    shutil.copyfile(INDEX, DRIVE_INDEX)
print("Index ready:", INDEX)

## Validate gold, inspect retrieved evidence, and confirm model availability
The coverage export is for human inspection, not an automatic equivalence claim.
It does not alter the annotations or provide gold passages to the generation run.

In [ ]:
inspection = """
from pathlib import Path
import json
from scripts.validate_gold import load_gold
from src.experiments.protocol import load_config, require_segmenter
from src.experiments.runner import pinned_segmenter
from scripts.build_fever_fts import FtsRetriever
config=load_config(Path('configs/evaluation_halueval_colab.yaml'))
with pinned_segmenter(require_segmenter(config)):
    gold=load_gold(Path('data/gold/pilot_dev.jsonl'),'dev')
with FtsRetriever(Path('indexes/fever_fts.sqlite')) as retriever:
    rows=[{'record_id':r.id,'claim_id':c.id,'claim':c.text,
           'retrieved':[p.__dict__ for p in retriever.retrieve(c.text,5)]}
          for r in gold for c in r.check_worthy_claims]
Path('results').mkdir(exist_ok=True)
Path('results/pilot_retrieval_review.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2))
print('Reviewed gold:',len(gold),'records;',sum(len(r.claims) for r in gold),'claims')
print('Index passages:',retriever.info['passage_count'])
from groq import Groq
model_ids={model.id for model in Groq().models.list().data}
assert 'llama-3.3-70b-versatile' in model_ids, 'Configured model unavailable: choose an available model before freezing the run.'
print('Configured model is available.')
"""
subprocess.run([PYTHON, "-c", inspection], check=True, cwd=REPO)
shutil.copyfile(REPO / "results/pilot_retrieval_review.json", DRIVE_ROOT / "pilot_retrieval_review.json")

## Execute pipeline + B0
Use a new output directory for changed code/settings. Set `RESUME=True` only to
resume exactly the same experiment. The output folder and caches live on Drive.

In [ ]:
RESUME = False
RUN_DIR = DRIVE_ROOT / "pilot_repair_v1"
command = [PYTHON, "scripts/run_benchmark.py",
           "--config", "configs/evaluation_halueval_colab.yaml",
           "--inputs", "data/gold/pilot_dev.jsonl", "--splits", "data/gold/pilot_splits.json",
           "--task", "repair", "--split", "dev", "--output-dir", str(RUN_DIR)]
if RESUME:
    command.append("--resume")
subprocess.run(command, check=True, cwd=REPO)
print("Execution saved:", RUN_DIR)

## Score reference agreement and export independent repair review
This is the next review stage: humans assess the newly generated repairs, rather
than re-reviewing original gold. A final repair report requires adjudicated output
ratings; reference similarity alone is not factual correctness.

In [ ]:
METRICS = DRIVE_ROOT / "pilot_reference_metrics.json"
if not METRICS.exists():
    subprocess.run([PYTHON,"scripts/run_eval.py","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--output",str(METRICS)],check=True,cwd=REPO)
PACKET = DRIVE_ROOT / "repair_review_v1"
if not PACKET.exists():
    subprocess.run([PYTHON,"scripts/review_repairs.py","export","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--packet-dir",str(PACKET)],check=True,cwd=REPO)
PROGRESS = DRIVE_ROOT / "pilot_progress_v1"
if not PROGRESS.exists():
    subprocess.run([PYTHON,"scripts/build_evaluation_report.py","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--mode","progress","--output-dir",str(PROGRESS)],check=True,cwd=REPO)
print("Give reviewers only:", PACKET / "reviewer")
print("After independent review/adjudication, follow docs/independent_evaluation.md to build the final pilot report.")